# NeqSim multi-process diagrams: block diagram, PFD and P&ID

This notebook demonstrates a **multi-area `ProcessModel`** at three levels:
**block diagram → PFD → P&ID proposal**.

The example uses two `ProcessSystem` areas:
1. inlet heating and HP separation,
2. gas compression, cooling, export scrubbing and export compression.

The same topology is progressively enriched rather than redrawn as unrelated figures.

In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
from IPython.display import SVG, display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neqsim>=3.20.0"], check=True)
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False)

NEQSIM_AVAILABLE = importlib.util.find_spec("neqsim") is not None
print("NeqSim available:", NEQSIM_AVAILABLE)

## 1. Build and run a multi-process NeqSim model

The HP-separator gas outlet is passed directly into the compression `ProcessSystem`,
so the `ProcessModel` contains a real **inter-area live stream**.

In [ ]:
if NEQSIM_AVAILABLE:
    from neqsim import jneqsim

    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    Stream = jneqsim.process.equipment.stream.Stream
    Heater = jneqsim.process.equipment.heatexchanger.Heater
    Cooler = jneqsim.process.equipment.heatexchanger.Cooler
    Separator = jneqsim.process.equipment.separator.Separator
    Compressor = jneqsim.process.equipment.compressor.Compressor
    ProcessSystem = jneqsim.process.processmodel.ProcessSystem
    ProcessModel = jneqsim.process.processmodel.ProcessModel

    fluid = SystemSrkEos(318.15, 100.0)
    for name, z in [
        ("methane", 0.82), ("ethane", 0.08), ("propane", 0.04),
        ("n-butane", 0.02), ("n-pentane", 0.015),
        ("n-hexane", 0.015), ("water", 0.01)
    ]:
        fluid.addComponent(name, z)
    fluid.setMixingRule("classic")

    feed = Stream("Feed", fluid)
    feed.setFlowRate(100000.0, "kg/hr")
    feed.setTemperature(45.0, "C")
    feed.setPressure(100.0, "bara")

    heater = Heater("E-101 Inlet heater", feed)
    heater.setOutTemperature(55.0, "C")
    hpsep = Separator("V-101 HP separator", heater.getOutletStream())

    separation = ProcessSystem("Inlet & separation")
    for unit in [feed, heater, hpsep]:
        separation.add(unit)

    comp1 = Compressor("K-201 1st-stage compressor", hpsep.getGasOutStream())
    comp1.setOutletPressure(125.0)
    comp1.setIsentropicEfficiency(0.78)

    cooler = Cooler("E-202 Aftercooler", comp1.getOutletStream())
    cooler.setOutTemperature(35.0, "C")
    scrubber = Separator("V-202 Export scrubber", cooler.getOutletStream())

    comp2 = Compressor("K-203 Export compressor", scrubber.getGasOutStream())
    comp2.setOutletPressure(150.0)
    comp2.setIsentropicEfficiency(0.78)

    compression = ProcessSystem("Gas compression")
    for unit in [comp1, cooler, scrubber, comp2]:
        compression.add(unit)

    plant = ProcessModel()
    plant.add("Inlet & separation", separation)
    plant.add("Gas compression", compression)
    plant.run()
    print("NeqSim ProcessModel executed.")
else:
    plant = None
    print("NeqSim runtime unavailable here; diagram demonstration still runs.")

## 2. Block diagram

A block diagram intentionally hides equipment detail and shows the process-area structure.

In [ ]:
block_dot = r'''
digraph G {
  graph [rankdir=LR, label="Multi-area ProcessModel — block diagram", labelloc=t, fontsize=20];
  node [shape=box, style="rounded", fontsize=12];
  feed [label="Feed / gathering"];
  sep [label="Area 1\nInlet & separation"];
  comp [label="Area 2\nGas compression"];
  gas [label="Sales gas"]; oil [label="Oil export"]; water [label="Produced water"];
  feed -> sep [label="wellstream"];
  sep -> comp [label="HP gas"];
  comp -> gas [label="export gas"];
  sep -> oil [label="liquid"];
  sep -> water [label="water"];
}
'''
Path("block.dot").write_text(block_dot)
subprocess.run(["dot","-Tsvg","block.dot","-o","block.svg"], check=True)
display(SVG("block.svg"))

## 3. Process Flow Diagram (PFD)

NeqSim's native multi-area API can produce a common plant graph:

```python
plant_dot = plant.toDOT()
```

For large models, `exportAreaDOT(...)` can also create one diagram per `ProcessSystem`.
The PFD adds individual equipment, stream identities and operating data.

In [ ]:
if NEQSIM_AVAILABLE:
    native_dot = str(plant.toDOT())
    Path("neqsim_native_processmodel.dot").write_text(native_dot)
    subprocess.run(["dot","-Tsvg","neqsim_native_processmodel.dot","-o","neqsim_native_processmodel.svg"], check=True)
    display(SVG("neqsim_native_processmodel.svg"))
else:
    pfd_dot = r'''
    digraph G {
      graph [rankdir=LR, label="PFD — multi-process NeqSim model", labelloc=t, fontsize=20, compound=true];
      node [fontsize=10];
      subgraph cluster_sep {
        label="ProcessSystem: Inlet & separation";
        feed [label="Feed",shape=ellipse]; heater [label="E-101\nInlet heater",shape=circle];
        sep [label="V-101\nHP separator",shape=cylinder];
        oil [label="Oil export",shape=ellipse]; water [label="Produced water",shape=ellipse];
        feed->heater [label="100 bara / 45 °C"]; heater->sep [label="95 bara / 55 °C"];
        sep->oil [label="oil"]; sep->water [label="water"];
      }
      subgraph cluster_comp {
        label="ProcessSystem: Gas compression";
        c1 [label="K-201\n1st-stage compressor",shape=trapezium];
        cool [label="E-202\nAftercooler",shape=circle];
        scrub [label="V-202\nExport scrubber",shape=cylinder];
        c2 [label="K-203\nExport compressor",shape=trapezium];
        sales [label="Sales gas",shape=ellipse];
        c1->cool->scrub->c2->sales;
      }
      sep->c1 [label="HP gas"];
    }
    '''
    Path("pfd.dot").write_text(pfd_dot)
    subprocess.run(["dot","-Tsvg","pfd.dot","-o","pfd.svg"], check=True)
    display(SVG("pfd.svg"))

## 4. P&ID proposal

A P&ID layers engineering intent on top of the process topology:
equipment tags, line IDs, control valves, instruments, signal paths and safeguarding.
Automated output must be treated as a **proposal requiring engineering review**, not an approved construction drawing.

Current NeqSim master includes semantic engineering-diagram and P&ID-synthesis infrastructure;
this compact teaching cell visualizes the same process with typical instrument/control additions.

In [ ]:
pid_dot = r'''
digraph G {
  graph [rankdir=LR, label="P&ID proposal — tagged lines, valves and control loops", labelloc=t, fontsize=20];
  node [fontsize=9];
  feed [label="FEED\n10\"-HC-1001",shape=ellipse];
  xv [label="XV-101",shape=diamond]; heat [label="E-101\nInlet heater",shape=circle];
  sep [label="V-101\nHP separator",shape=cylinder];
  lv [label="LV-101",shape=diamond]; oil [label="OIL EXPORT\n8\"-HC-1101",shape=ellipse];
  pv [label="PV-201",shape=diamond]; c1 [label="K-201\n1st-stage compressor",shape=trapezium];
  cool [label="E-202\nAftercooler",shape=circle]; scrub [label="V-202\nExport scrubber",shape=cylinder];
  c2 [label="K-203\nExport compressor",shape=trapezium]; gas [label="SALES GAS\n10\"-HC-2201",shape=ellipse];
  feed->xv->heat->sep; sep->lv->oil; sep->pv->c1->cool->scrub->c2->gas;

  lt [label="LT-101",shape=circle]; lic [label="LIC-101",shape=circle,style=dashed];
  pt [label="PT-201",shape=circle]; pic [label="PIC-201",shape=circle,style=dashed];
  tt [label="TT-202",shape=circle]; tic [label="TIC-202",shape=circle,style=dashed];

  sep->lt [style=dotted,arrowhead=none]; lt->lic [style=dashed]; lic->lv [style=dashed,label="signal"];
  sep->pt [style=dotted,arrowhead=none]; pt->pic [style=dashed]; pic->pv [style=dashed,label="signal"];
  cool->tt [style=dotted,arrowhead=none]; tt->tic [style=dashed]; tic->cool [style=dashed,label="duty control"];
}
'''
Path("pid.dot").write_text(pid_dot)
subprocess.run(["dot","-Tsvg","pid.dot","-o","pid.svg"], check=True)
display(SVG("pid.svg"))

## 5. Summary

| Diagram | Main question answered | Typical NeqSim source |
|---|---|---|
| Block diagram | How are process areas connected? | `ProcessModel` area/inter-area topology |
| PFD | What equipment and streams perform the process? | `ProcessSystem` / `ProcessModel.toDOT()` |
| P&ID proposal | How could the process be instrumented and controlled? | process topology + engineering/P&ID metadata/rules |

A natural next notebook is a **five-area facility model**:
inlet separation → recompression → dehydration → dew-point/NGL recovery → export,
with one common PFD, per-area PFDs and a multi-sheet P&ID proposal.